# [F7: Anomalies sur le S&P 500]

**Nom et prénom :** Touassi Saad      **Code Apogée :** 23009123     **Groupe :** G3     **Filière :** S7 FIN     **Date :** 11/17/2026

**Dépôt GitHub :** https://github.com/touassisaad7/23009123-Touassi-Saad-G3.git

## 1. Sujet et problématique
- **Intérêt du sujet** (2 à 3 lignes) : La détection précoce ou la caractérisation des mouvements extrêmes sur les indices boursiers majeurs comme le S&P 500 est essentielle pour la gestion des risques de marché, l'allocation d'actifs et la couverture de portefeuille (hedging) par les gestionnaires de fonds et les directions des risques.
- **Problématique** (une question) : Un algorithme de détection d'anomalies non supervisé permet-il d'identifier automatiquement les journées de stress financier et les chocs de marché historiques du S&P 500 à partir de ses rendements et de sa volatilité récente ?
- **Objectif mesurable** (métrique, référence, période de test, seuil) :

 -*Métrique principale* : Nombre d’événements de stress documentés (liste de référence de 6 chocs majeurs listés ci-dessous, détectés par une alerte dans une fenêtre de $\pm 2$ jours).
  
  -*Métrique secondaire* : Part de jours signalés (taux de faux positifs/contamination).
  
  -*Référence* : Règle des $3\sigma$ sur le rendement log journalier (paramètres $\mu$ et $\sigma$ calculés strictement sur l'apprentissage).
  
-*Période* : Apprentissage du début de série au 31/12/2019 ; Test du 01/01/2020 au 31/10/2026.
  
  -*Seuil de succès* : L'Isolation Forest détecte au moins autant d’événements de référence que la règle $3\sigma$, tout en maintenant une part de jours signalés inférieure ou égale à 5 %.
- **Nature du problème** (et pourquoi) : Détection d'anomalies (apprentissage non supervisé). Ce choix est justifié par l'absence de variable cible étiquetée « anomalie » dans les séries financières brutes et par la rareté des événements extrêmes.
- **Algorithme principal envisagé** (n° de l'annexe A) : N° 24 — Isolation Forest (`IsolationForest` de scikit-learn).





## 2. Jeu de données

- **Nom, source exacte et lien d'accès** : Indice boursier S&P 500 (`SP500`), extrait depuis la base de données économique **FRED** (Federal Reserve Bank of St. Louis) à l'adresse : https://fred.stlouisfed.org/series/SP500.
- **Date d'extraction** : 07/11/2026 (dernière cotation disponible au 05/11/2026).
- **Conditions d'utilisation / Licence** : Données publiques diffusées par S&P Dow Jones Indices LLC via FRED ; utilisation autorisée à des fins éducatives et de recherche non commerciale (rediffusion soumise aux droits de propriété intellectuelle de S&P Dow Jones Indices)[cite: 1].
- **Taille et fréquence** : 2 609 observations brutes et 2 colonnes (`observation_date`, `SP500`)[cite: 1], en fréquence quotidienne (jours ouvrés du lundi au vendredi), couvrant exactement 10 ans du **07/11/2016** au **05/11/2026**[cite: 1].
- **Unité d'observation** : Une journée de cotation à la Bourse de New York (niveau de clôture quotidien de l'indice S&P 500).
- **Variable cible** : Aucune variable cible native, le problème étant formulé en apprentissage non supervisé (détection d'anomalies). Les alertes générées sont évaluées *a posteriori* face à la liste des 6 événements de stress documentés fixée en section 1.
- **Adéquation** : L'indice S&P 500 reflète directement la valorisation des 500 plus grandes entreprises américaines cotées ; la période 2016–2026 permet d'entraîner le modèle sur un régime de marché relativement calme (2016–2019) puis de tester sa réactivité face à des chocs systémiques majeurs (krach Covid-19 de mars 2020 et choc inflationniste de 2022).
- **Limites** :
  1. Fenêtre historique limitée aux 10 dernières années sur FRED, excluant les crises antérieures (2000, 2008).
  2. Changement de régime structurel important entre la période d'apprentissage (taux bas, faible volatilité) et la période de test (pandémie, hausse rapide des taux directeurs).
  3. Présence de 96 valeurs manquantes (3,68 %) correspondant aux jours fériés américains (marchés fermés en semaine), ainsi que l'absence des prix intrajournaliers (Open, High, Low) et des volumes d'échange.

### Dictionnaire des variables (brutes et construites)

| Variable | Description | Type | Unité | Rôle | Manquants (%) |
| --- | --- | --- | --- | --- | --- |
| `observation_date` | Date de la séance boursière | Date | AAAA-MM-JJ | Identifiant temporel | 0,00 % |
| `SP500` | Niveau de clôture de l'indice S&P 500 | Numérique | Points d'indice | Brute (base) | 3,68 % (96 jours fériés) |
| `rendement` | Rendement logarithmique quotidien : $\ln(P_t / P_{t-1})$ | Numérique | Sans unité (ratio) | Explicative | 0,04 % (1ère ligne après ffill) |
| `rendement_abs` | Amplitude absolue du mouvement quotidien : $\|r_t\|$ | Numérique | Sans unité (ratio) | Explicative | 0,04 % (1ère ligne après ffill) |
| `volatilite_20j` | Écart-type glissant des rendements sur les 20 derniers jours ouvrés | Numérique | Sans unité (ratio) | Explicative | 0,77 % (20 premières lignes) |
| `retard_1` | Rendement logarithmique de la veille : $r_{t-1}$ | Numérique | Sans unité (ratio) | Explicative | 0,08 % (2 premières lignes) |

In [ ]:
import pandas as pd
from pathlib import Path
import urllib.request

# Définition et création du dossier de données
DATA_RAW = Path("data/raw")
DATA_RAW.mkdir(parents=True, exist_ok=True)

chemin_donnees = DATA_RAW / "SP500.csv"

# Téléchargement automatique du fichier S&P 500 depuis FRED s'il n'est pas présent
if not chemin_donnees.exists():
    print("Téléchargement des données S&P 500 depuis FRED...")
    url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=SP500"
    urllib.request.urlretrieve(url, chemin_donnees)
    print("Téléchargement terminé.")

# Chargement du fichier brut
df_raw = pd.read_csv(chemin_donnees)

# Conversion des types
df_raw["observation_date"] = pd.to_datetime(df_raw["observation_date"])
df_raw["SP500"] = pd.to_numeric(df_raw["SP500"], errors="coerce")
df_raw = df_raw.sort_values("observation_date").reset_index(drop=True)

# Vérifications obligatoires
nb_lignes, nb_cols = df_raw.shape
date_min = df_raw["observation_date"].min().date()
date_max = df_raw["observation_date"].max().date()
nb_na = df_raw["SP500"].isna().sum()
pct_na = df_raw["SP500"].isna().mean() * 100

print(f"Dimensions du jeu brut : {nb_lignes} lignes, {nb_cols} colonnes")
print(f"Plage temporelle : du {date_min} au {date_max} (fréquence quotidienne ouvrée)")
print(f"Valeurs manquantes sur SP500 : {nb_na} ({pct_na:.2f} %)")
print("\nTypes des variables :")
print(df_raw.dtypes)
display(df_raw.head())

Dimensions du jeu brut : 2609 lignes, 2 colonnes
Plage temporelle : du 2016-11-07 au 2026-11-05 (fréquence quotidienne ouvrée)
Valeurs manquantes sur SP500 : 96 (3.68 %)

Types des variables :
observation_date    datetime64[ns]
SP500                      float64
dtype: object


,observation_date,SP500
0,2016-11-07,2131.52
1,2016-11-08,2139.56
2,2016-11-09,2163.26
3,2016-11-10,2167.48
4,2016-11-11,2164.45
